In [1]:
import sys
print(" Your Python:", sys.executable)
!{sys.executable} -m pip install --upgrade pip openpyxl --force-reinstall
print(" openpyxl installed to Python 3.13!")


 Your Python: c:\Users\Dhanyashree\AppData\Local\Programs\Python\Python313\python.exe
^C
 openpyxl installed to Python 3.13!


In [ ]:
import pandas as pd
df = pd.read_excel('datafinal.xlsx', sheet_name='Sheet1')
print(" Shape:", df.shape)
df.head()


📁 Shape: (201, 9)


,share-of-plastic-waste-that-is-mismanaged,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,Entity,Code,Year,Total population,Coastal population,Coastal population density(/km^2),% mismanaged from total regional waste,Mismanaged waste(tonnes),Oceans plastic leakage (y)
1,Americas (excl. USA),NaN,2000,239839213,170000000,77.27,32.366478,5291803,1104799
2,Americas (excl. USA),NaN,2001,244346221,171700000,78.05,32.316288,5561646,1198096
3,Americas (excl. USA),NaN,2002,248656910,173300000,78.77,32.274937,5928127,1298042
4,Americas (excl. USA),NaN,2003,252779849,175000000,79.55,32.21104,6267913,1404860


In [ ]:
# Fix messy Excel header + units
df = df.iloc[1:].reset_index(drop=True)  # Skip first junk row
df.columns = ['Entity','Code','Year','Total_pop','Coastal_pop','Density',
              'MWI_pct','Mismanaged_tonnes','Target_MT']

# Fix units (your project requirement!)
df['MWI'] = df['MWI_pct'] / 100  # % → 0-1 scale
df['Target_MT'] = pd.to_numeric(df['Target_MT'], errors='coerce')

X = df[['Density', 'MWI']].dropna()  # Inputs
y = df['Target_MT'].loc[X.index]     # Target (metric tons)

print(f" READY: {len(X)} rows for training")
print("\n Sample inputs:")
print(X.head())
print(f"\n Sample targets: {y.head().tolist()}")


✅ READY: 200 rows for training

📊 Sample inputs:
  Density       MWI
0   77.27  0.323665
1   78.05  0.323163
2   78.77  0.322749
3   79.55   0.32211
4   80.32  0.321449

🎯 Sample targets: [1104799.0, 1198096.0, 1298042.0, 1404860.0, 1519021.0]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.9, random_state=42)

# 100 smart trees!
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Train R² (how well model memorized training data)
y_train_pred = model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
print(f" Train R²: {train_r2:.3f}")

# Test R² (your existing code)
y_pred = model.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
print(f" Test R²: {test_r2:.3f}")

# Results
y_pred = model.predict(X_test)
print(f" Test R² Score: {r2_score(y_test, y_pred):.3f}")
print(f" Mean Error: {mean_absolute_error(y_test, y_pred):,.0f} tons")
print(f" Density importance: {model.feature_importances_[0]:.1%}")
print(f" MWI importance: {model.feature_importances_[1]:.1%}")


📚 Train R²: 0.912
🎯 Test R²: 0.833
🎯 Test R² Score: 0.833
📏 Mean Error: 1,466,889 tons
🌟 Density importance: 27.9%
🌟 MWI importance: 72.1%


In [ ]:
import matplotlib.pyplot as plt

#  Predict NEW data
new_data = [[1000, 0.3]]  # density=1000/km², MWI=30%
prediction = model.predict(new_data)[0]
print(f" NEW Prediction: {prediction:,.0f} metric tons ocean plastic")

# Plot real vs predicted
plt.figure(figsize=(10,6))
plt.scatter(y_test, y_pred, alpha=0.7, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Real Ocean Plastic (metric tons)')
plt.ylabel('Predicted Ocean Plastic (metric tons)')
plt.title(f'Your Model: R² = {r2_score(y_test, y_pred):.3f}')
plt.grid(True, alpha=0.3)
plt.show()
